# <font color='264CC7'> activity_tracker </font>

Este notebook tiene como objetivo desarrollar y validar progresivamente los módulos que conforman el sistema telegram-notion-activity-tracker. Cada sección corresponde a un componente funcional independiente, que será probado de manera aislada antes de integrarse en el flujo completo.

El sistema estará compuesto por los siguientes módulos:
- Clasificador: analiza el texto de la actividad y lo categoriza según la taxonomía definida.
- Transcriptor: convierte mensajes de voz en texto procesable.
- Envío a Notion: construye el registro estructurado y lo inserta en la base de datos.
- Bot de Telegram: recibe mensajes y audios, y activa el flujo de procesamiento.
- Pipeline: integra todos los módulos en un proceso completo desde la captura hasta el almacenamiento.

Los paquetes necesarios son:

In [1]:
# !pip install pymupdf 
# !pip install openai
# !pip install notion-client
# !pip install pyTelegramBotAPI

---
## <font color='264CC7'> Clasificador </font>


In [1]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field
import json

In [2]:
# Configurar la clave de API
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
def load_categories(file_path: str) -> dict:
    with open(file_path, 'r', encoding='utf-8') as file:
        categories = json.load(file)
    return categories

categories = load_categories('categories.json')
print(categories)

{'Carreras-Programas': {'nombre': 'Diseño, seguimiento y evaluación de carreras y programas', 'descripcion': 'Actividades relacionadas con la planificación, monitoreo y evaluación de las carreras y programas de la Facultad, incluyendo generación de lineamientos generales.'}, 'Internacionalizacion': {'nombre': 'Internacionalización curricular y movilidad docente', 'descripcion': 'Acciones orientadas a fortalecer la dimensión internacional del currículo y promover la movilidad académica docente.'}, 'Recursos': {'nombre': 'Optimización de los recursos didácticos y de la información para el aprendizaje', 'descripcion': 'Gestión y mejora de materiales didácticos, repositorios académicos y recursos de información para apoyar el proceso de enseñanza-aprendizaje.'}, 'RdA': {'nombre': 'Medición de los resultados de aprendizaje', 'descripcion': 'Diseño, aplicación y análisis de instrumentos y estrategias para evaluar el logro de los resultados de aprendizaje, incluye rutas académicas y lo realci

In [4]:
class Activity(BaseModel):
    name: str
    description: str
    date: str = Field(description="Fecha en formato AAAA-MM-DD.")
    category: str

class ActivityBatch(BaseModel):
    activities: list[Activity] = Field(description="Lista de actividades detectadas. Puede contener una o más actividades.")

def classify_activities(texto: str, categories: dict, today: str) -> list[Activity]:
    allowed = list(categories.keys())

    system_prompt = (
        "Eres un asistente para registrar actividades de gestión académica de la USDIC en la FCENA (PUCE). "
        "Contexto institucional: FCENA incluye las carreras de Biología, Microbiología, Química, Bioingeniería, "
        "Ciencias Biomédicas y Ciencia de Datos; y programas de posgrado de Maestría en Biología y "
        "Maestría en Ciencias Actuariales. "
        "Conceptos clave:\n"
        "- Rutas académicas: procesos para que un estudiante apruebe un resultados de aprendizaje (RdA) pendiente.\n"
        "- Banner: sistema de gestión académica utilizado para matrícula, registro de notas, programación académica.\n"
        "- EVA: Entorno Virtual de Aprendizaje (aulas virtuales).\n"
        "Tu tarea es estructurar y clasificar actividades relacionadas con estos ámbitos. "
        "No inventes información; si algo no está explícito, manténlo general."
    )
    user_prompt = (
        "1) Detecta si el texto contiene una o varias actividades.\n"
        "2) Para cada actividad, elige EXACTAMENTE una categoría de la lista permitida.\n"
        "3) Para cada actividad, genera un nombre corto y simple pero que capture lo que se hace (3–8 palabras).\n"
        "4) Para cada actividad, redacta una descripción breve (1 frases).\n"
        f"5) Si no hay fecha explícita, usa la de hoy que es {today} (formato AAAA-MM-DD).\n"
        "6) Devuelve solo el objeto del esquema con 'activities' como lista. Si hay una sola actividad, igual devuélvela en la lista.\n\n"
        f"Categorías permitidas: {allowed}\n\n"
        f"Descripción de las categorías:\n" + "\n".join([f"- {key}: {value['descripcion']}" for key, value in categories.items()]) + "\n"
        "No inventes categorías. Si ninguna aplica, elige la más cercana evita lo máximo en usar la categoría Otras.\n\n"
        f"Actividad: {texto}"
    )
    response = client.responses.parse(
        model="gpt-4.1-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        text_format=ActivityBatch,
        temperature=0,
    )
    parsed = response.output_parsed
    if not parsed or not parsed.activities:
        raise ValueError("No se detectaron actividades válidas en la respuesta del modelo.")
    return parsed.activities

In [ ]:
# texto = "Ayer generé los archivos de horarios de las diferentes carreras de la facultad que fueron enviado a estudiantes y a la DDE."

# actividad_clasificada = classify_activities(texto, categories, "2026-20-02")
# print(actividad_clasificada)

[Activity(name='Generación y envío de horarios', description='Se generaron los archivos de horarios de las diferentes carreras y se enviaron a estudiantes y a la DDE.', date='2026-02-20', category='Programacion')]


In [7]:
# texto = "Solicione un inconveniente reportado por estudiantes en el que no se habían pasado las notas de los itinerarios al sistema Banner"

# actividad_clasificada = classify_activity(texto, categories, "2026-20-02")
# print(actividad_clasificada)

---
## <font color='264CC7'> Transcriptor </font>

In [8]:
import os
from pathlib import Path
from openai import OpenAI

In [9]:
# Configurar la clave de API
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [10]:
def transcribe_audio(audio_path: str) -> str:
    p = Path(audio_path)
    if not p.exists():
        raise FileNotFoundError(f"No existe el archivo: {audio_path}")

    transcription = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe", 
        file=p.open("rb")
    )

    return transcription.text


In [11]:
# txt = transcribe_audio("datos/Audio.m4a")
# print("El texto transcrito es:", txt)

---
## <font color='264CC7'> Envío a Notion </font>

In [6]:
from pydantic import BaseModel, Field
import os
from notion_client import Client as NotionClient

In [7]:
NOTION_DATABASE_ID = os.getenv("NOTION_DATABASE_ID")
notion = NotionClient(auth=os.getenv("NOTION_TOKEN"))

In [8]:
class Activity(BaseModel):
    name: str
    description: str
    date: str = Field(description="Fecha en formato AAAA-MM-DD.")
    category: str

class ActivityBatch(BaseModel):
    activities: list[Activity] = Field(description="Lista de actividades detectadas. Puede contener una o más actividades.")

# Defino una Activity 
actividad = Activity(
    name='Generación y envío de horarios',
    description='Se generaron los archivos de horarios de las carreras y se enviaron a estudiantes y a la DDE.',
    date='2026-02-19',
    category='Programacion'
)
actividad_batch = ActivityBatch(activities=[actividad])

In [9]:
def enviar_a_notion(actividad: Activity):
    propiedades = {
        "Actividad": {"title": [{"text": {"content": actividad.name}}]},
        "Categoría": {"select": {"name": actividad.category}},
        "Fecha": {"date": {"start": actividad.date}},
        "Descripción": {"rich_text": [{"text": {"content": actividad.description}}]}
    }
    try:
        notion.pages.create(
            parent={"database_id": NOTION_DATABASE_ID},
            properties=propiedades
        )
    except Exception as e:
        print(f"Error al enviar a Notion: {e}")
        return {"ok": False, "error": str(e)}

    return {"ok": True}

def enviar_varias_a_notion(actividades: list[Activity]):
    resultados = []
    errores = []

    for actividad in actividades:
        resultado = enviar_a_notion(actividad)
        resultados.append(resultado)
        if not resultado.get("ok"):
            errores.append({"activity": actividad.name, "error": resultado.get("error", "Error desconocido")})

    return {
        "ok": len(errores) == 0,
        "total": len(actividades),
        "guardadas": len(actividades) - len(errores),
        "errores": errores,
    }


In [10]:
enviar_varias_a_notion(actividad_batch.activities)

{'ok': True, 'total': 1, 'guardadas': 1, 'errores': []}

---
## <font color='264CC7'> Bot de Telegram </font>

In [17]:
import telebot
from datetime import datetime

In [18]:
USUARIO_AUTORIZADO = int(os.getenv("USUARIO_AUTORIZADO"))
bot = telebot.TeleBot(os.getenv("TELEGRAM_TOKEN"))

In [ ]:
def es_usuario_autorizado(message) -> bool:
    return str(message.from_user.id) == str(USUARIO_AUTORIZADO)


def acceso_restringido(func):
    def wrapper(message):
        if not es_usuario_autorizado(message):
            bot.reply_to(message, "⛔ No estás autorizado para usar este bot.")
            return
        return func(message)
    return wrapper


def _fecha_hoy() -> str:
    return datetime.now().strftime("%Y-%m-%d")


@bot.message_handler(commands=["start"])
@acceso_restringido
def start(message):
    bot.send_message(
        message.chat.id,
        "👋 ¡Hola! Cuéntame qué actividades realizaste hoy"
    )


@bot.message_handler(content_types=["text"])
@acceso_restringido
def handle_text(message):
    texto = (message.text or "").strip()
    if not texto:
        bot.reply_to(message, "Envía una descripción de la actividad (texto).")
        return

    bot.send_message(message.chat.id, "📌 Registrando actividad...")

    try:
        activities = classify_activities(
            texto=texto,
            categories=categories,
            today=_fecha_hoy()
        )
        result = enviar_varias_a_notion(activities)
        if result.get("ok"):
            bot.send_message(message.chat.id, f"✅ {result.get('guardadas', 0)} de {result.get('total', 0)} actividades registradas en Notion.")
            for activity in activities:
                bot.send_message(
                    message.chat.id,
                    "✅ Actividad registrada en Notion: \n"
                    f"📌 Nombre: {activity.name}\n"
                    f"📌 Categoría: {activity.category}\n"
                )
        else:
            bot.send_message(message.chat.id, f"⚠️ No se pudo registrar: {result.get('error', 'Error desconocido')}")
    except Exception as e:
        bot.send_message(message.chat.id, f"⚠️ Ocurrió un error: {str(e)}")


@bot.message_handler(content_types=["voice"])
@acceso_restringido
def handle_voice(message):
    bot.send_message(message.chat.id, "🎙️ Recibido. Transcribiendo y registrando...")

    file_info = bot.get_file(message.voice.file_id)
    downloaded = bot.download_file(file_info.file_path)

    os.makedirs("temp", exist_ok=True)
    ruta_audio = os.path.join("temp", f"voice_{message.message_id}.ogg")
    with open(ruta_audio, "wb") as f:
        f.write(downloaded)

    try:
        texto = transcribe_audio(
            audio_path=ruta_audio
        )
        activities = classify_activities(
            texto=texto,
            categories=categories,
            today=_fecha_hoy()
        )
        result = enviar_varias_a_notion(activities)
        if result.get("ok"):
            bot.send_message(
                message.chat.id,
                f"✅ {result.get('guardadas', 0)} de {result.get('total', 0)} actividades registradas en Notion."
            )
            for activity in activities:
                bot.send_message(
                    message.chat.id,
                    "✅ Actividad registrada en Notion: \n"
                    f"📌 Nombre: {activity.name}\n"
                    f"📌 Categoría: {activity.category}\n"
                )
        else:
            bot.send_message(message.chat.id, f"⚠️ No se pudo registrar: {result.get('error', 'Error desconocido')}")
    except Exception as e:
        bot.send_message(message.chat.id, f"⚠️ Ocurrió un error: {str(e)}")
    finally:
        try:
            os.remove(ruta_audio)
        except OSError:
            pass


@bot.message_handler(func=lambda message: True)
def fallback(message):
    if not es_usuario_autorizado(message):
        return
    bot.reply_to(message, "Envía una actividad como texto o una nota de voz.")


# Iniciar el bot
bot.polling(none_stop=True)